In [6]:
import os

!pip install -q qiskit-ibm-runtime

from qiskit_ibm_runtime import QiskitRuntimeService
from google.colab import userdata

# Authenticate with IBM Quantum (using Colab secrets for the token)
# Ensure your IBMQ_TOKEN is set in Colab secrets
service = None
try:
    # Attempt to retrieve the token from Colab secrets
    ibm_q_token = userdata.get('IBMQ_TOKEN')
    if ibm_q_token:
        QiskitRuntimeService.save_account(
            token=ibm_q_token, channel='ibm_quantum_platform', overwrite=True
        )
        print("IBM Quantum account saved successfully.")
        service = QiskitRuntimeService(channel='ibm_quantum_platform')
        print("Connected to IBM Quantum service.")
    else:
        print("Error: IBMQ_TOKEN not found in Colab secrets. Please add your token.")
except Exception as e:
    print(f"Could not save or connect to IBM Quantum account. Error: {e}")
    print("Please ensure your IBMQ_TOKEN is correctly set in Colab secrets.")


IBM Quantum account saved successfully.


qiskit_runtime_service.__init__:WARNING:2026-09-17 20:03:35,832: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Aura-9_Oracle_Alpha. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


Connected to IBM Quantum service.


In [10]:
import datetime

# Retrieve the last 3 jobs
print("Retrieving jobs from today...")

# Check if 'service' is defined and not None
if 'service' in globals() and service is not None:
    # Get all jobs (or a larger limit if needed to ensure today's jobs are included)
    all_jobs = service.jobs(limit=100) # Increased limit to ensure capturing all recent jobs

    today = datetime.date(2026, 9, 17) # Explicitly setting today to Sept 17th as requested
    today_jobs = [job for job in all_jobs if job.creation_date.date() == today]

    if today_jobs:
        print(f"Found {len(today_jobs)} jobs created today ({today}):")
        for i, job in enumerate(today_jobs):
            print(f"\nJob {i+1}:")
            print(f"  Job ID: {job.job_id()}")
            print(f"  Status: {job.status()}")
            print(f"  Creation Date: {job.creation_date}")
            # You can fetch more details if needed, e.g., job.result() once it's done
    else:
        print(f"No jobs found created today ({today}).")
else:
    print("Error: The 'service' object was not initialized. Please ensure the previous cell (authentication) ran successfully before running this cell.")

Retrieving jobs from today...


qiskit_runtime_service._create_backend_obj:WARNING:2026-09-17 20:12:14,259: Unable to create configuration for ibm_torino. '404 Client Error: Not Found for url: https://quantum.cloud.ibm.com/api/v1/backends/ibm_torino/configuration. {"errors":[{"code":"not_found","message":"device not found","more_info":"https://cloud.ibm.com/apidocs/quantum-computing#error-handling"}],"trace":"13a0e3a2-d384-4e35-8c89-88ed42701639"}\n' 
qiskit_runtime_service._create_backend_obj:WARNING:2026-09-17 20:12:14,743: Unable to create configuration for ibm_torino. '404 Client Error: Not Found for url: https://quantum.cloud.ibm.com/api/v1/backends/ibm_torino/configuration. {"errors":[{"code":"not_found","message":"device not found","more_info":"https://cloud.ibm.com/apidocs/quantum-computing#error-handling"}],"trace":"45224841-0bf1-4773-9ea8-1b06494c928a"}\n' 
qiskit_runtime_service._create_backend_obj:WARNING:2026-09-17 20:12:15,291: Unable to create configuration for ibm_torino. '404 Client Error: Not Found 

Found 2 jobs created today (2026-09-17):

Job 1:
  Job ID: dalltmgpqrnc7396arfg
  Status: DONE
  Creation Date: 2026-09-17 03:30:34.248183+00:00

Job 2:
  Job ID: dallsu02fm4c73f1pl5g
  Status: DONE
  Creation Date: 2026-09-17 03:28:56.259409+00:00


In [14]:
# Get the result of one of the completed jobs

if today_jobs:
    # Assuming we want to get the result of the second job in the list
    target_job = today_jobs[1]
    print(f"Attempting to retrieve result for Job ID: {target_job.job_id()}")

    try:
        # Check if the job is indeed done before trying to get results
        if target_job.status() == 'DONE':
            result = target_job.result()
            print("Job result:")
            display(result) # Use display for better output formatting
        else:
            print(f"Job {target_job.job_id()} is not yet DONE. Current status: {target_job.status()}")
    except Exception as e:
        print(f"Error retrieving result for job {target_job.job_id()}: {e}")
else:
    print("No completed jobs found today to retrieve results from.")

Attempting to retrieve result for Job ID: dallsu02fm4c73f1pl5g
Job result:


PrimitiveResult([SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=5>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-09-17 03:28:59', stop='2026-09-17 03:29:01', size=4096>)])}, 'version': 2})

### Extracting Data from PrimitiveResult (Sampler)

Since your job resulted in a `SamplerPubResult`, the primary output is a `DataBin` containing a `BitArray` of measurement outcomes. We can convert these outcomes into a dictionary of counts.

In [13]:
# Access the SamplerPubResult from the PrimitiveResult
sampler_result = result[0]

# Access the DataBin which contains the BitArray
data_bin = sampler_result.data

# The BitArray of measurements is usually under an attribute like 'c' (for classical bits)
bit_array = data_bin.c

print("Raw BitArray (first few entries):\n", bit_array[:5])

# Convert the BitArray to counts
# The .get_counts() method is available on the BitArray or directly on SamplerPubResult in some Qiskit versions
# Let's try converting the BitArray first
try:
    counts = bit_array.get_counts()
    print("\nCounts (first 10 entries):\n", dict(list(counts.items())[:10]))
except AttributeError:
    # If get_counts is not directly on BitArray, it might be on the result itself or require manual processing
    print("\nBitArray does not have .get_counts() directly. Manual conversion:")
    from collections import Counter
    # The BitArray stores integer representations of bitstrings
    # To get actual bitstrings, we need to know num_bits
    num_bits = bit_array.num_bits # Assuming num_bits is available
    counts_raw = Counter(bit_array.array)
    counts = {
        format(k, '0' + str(num_bits) + 'b'): v
        for k, v in counts_raw.items()
    }
    print("\nCounts (first 10 entries, converted to bitstrings):\n", dict(list(counts.items())[:10]))


# Display the full counts (optional, for smaller results)
display(counts)


Raw BitArray (first few entries):
 BitArray(<shape=(), num_shots=5, num_bits=5>)

Counts (first 10 entries):
 {'11010': 1806, '10010': 144, '00010': 401, '01010': 548, '01110': 446, '00000': 99, '10101': 46, '10110': 112, '00011': 6, '11110': 167}


### Understanding Quasi-probabilities

Quasi-probabilities are typically associated with the **Estimator** primitive in Qiskit Runtime, not the Sampler. An Estimator job would return expectation values for observables, often represented as `EstimatorPubResult` objects, which might contain attributes like `quasi_dists` (for quasi-probability distributions) or `values` (for expectation values).

If you were using an Estimator, you would access them similarly, for example:

```python
# Assuming 'estimator_result' is an EstimatorPubResult
# quasi_dist = estimator_result.quasi_dists[0] # Access the first quasi-probability distribution
# print(quasi_dist)
```

Since your current job was a `Sampler`, the output is focused on measurement counts.